# Enhanced Image Generation with Real-ESRGAN (Hugging Face API)
This notebook uses the Hugging Face `gradio_client` to communicate with the `nightmareai/real-esrgan` Space.
Real-ESRGAN is a pure image upscaler. It does not accept text prompts, so we configure it directly to maximize clarity and quality based on the input image.

In [1]:
# Install dependencies
!pip install -q gradio_client ipywidgets certifi

In [ ]:
import os
import time
import shutil
import certifi
# Fix for Anaconda SSL FileNotFoundError on Windows
os.environ['SSL_CERT_FILE'] = certifi.where()

try:
    from gradio_client import Client, handle_file
except ImportError:
    print("Please run the cell above to install gradio_client")
    raise

# Replace with your actual Hugging Face API Key (It must start with 'hf_')
HF_API_KEY = 'API_KEY'

print("Initializing Real-ESRGAN Hugging Face Client...")
try:
    client = Client("Nick088/Real-ESRGAN_Pytorch", token=HF_API_KEY)
except Exception as e:
    print(f"Failed to connect to Hugging Face space: {e}")
    raise


Initializing Real-ESRGAN Hugging Face Client...
Loaded as API: https://nick088-real-esrgan-pytorch.hf.space


In [3]:
input_directory = os.path.join('Vehicle Snapshots', 'Bikes (Number-Plates On The Rear Mud-Guard)')
output_directory = os.path.join('Vehicle Snapshots', 'Enhanced Bike Snapshots')
os.makedirs(output_directory, exist_ok=True)

print(f"Scanning Directory: {input_directory}...")

if not os.path.isdir(input_directory):
    print(f"Directory {input_directory} not found. Please check the path.")
else:
    for root, dirs, files in os.walk(input_directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                file_path = os.path.join(root, file)
                new_filename = f"recreated_{file}"
                save_path = os.path.join(output_directory, new_filename)
                
                if os.path.exists(save_path):
                    print(f"Skipping {file_path}, enhanced image already exists.")
                    continue

                print(f"Loading {file_path}...")
                
                try:
                    max_retries = 3
                    recreated_image_path = None
                    
                    for attempt in range(max_retries):
                        try:
                            # Sending the image to Real-ESRGAN via Gradio Client
                            # The default predict method takes the image file.
                            result = client.predict(
                                img=handle_file(file_path),
                                api_name="/predict"
                            )
                            
                            # Gradio client returns a tuple or the path of the downloaded processed image
                            if isinstance(result, tuple):
                                recreated_image_path = result[0]
                            else:
                                recreated_image_path = result
                            
                            break
                                    
                        except Exception as e:
                            if attempt < max_retries - 1:
                                print(f"  -> Request error ({e}). Retrying attempt {attempt + 1} after 10 seconds...")
                                time.sleep(10)
                            else:
                                raise e
                    
                    if recreated_image_path and os.path.exists(recreated_image_path):
                        # Copy the temporary downloaded file to our final directory
                        shutil.copy(recreated_image_path, save_path)
                        print(f"Successfully upscaled and saved image to: {save_path}")
                    else:
                        print(f"Failed to generate image data for {file}")
                    
                    # RATE LIMIT BACKOFF FOR 10 RPM:
                    print("Waiting 6.5 seconds to respect the API limits...")
                    time.sleep(6.5)
                    
                except Exception as e:
                    print(f"Failed to process {file_path}: {e}")

print("Finished scanning and upscaling files.")

Scanning Directory: Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00217_bike.jpg...
Successfully upscaled and saved image to: Vehicle Snapshots\Enhanced Bike Snapshots\recreated_T00217_bike.jpg
Waiting 6.5 seconds to respect the API limits...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00272_bike.jpg...
Successfully upscaled and saved image to: Vehicle Snapshots\Enhanced Bike Snapshots\recreated_T00272_bike.jpg
Waiting 6.5 seconds to respect the API limits...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00304_bike.jpg...
Successfully upscaled and saved image to: Vehicle Snapshots\Enhanced Bike Snapshots\recreated_T00304_bike.jpg
Waiting 6.5 seconds to respect the API limits...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00336_bike.jpg...
Successfully upscaled and saved image to: Vehicle Snapshots\Enhanced Bike Snapshot